# Assignment 06: Clinic Data Wrangling Pipeline (Midterm)

Follow `README.md`: it names every file to save in `output/`, with its first line and line count. Replace each **TODO**, then restart the kernel and **Run All** before you commit, so the saved outputs show your work.

In [127]:
# Supplied setup: run this cell first and leave it unchanged.
from pathlib import Path
import hashlib
import json
import subprocess
import sys

import numpy as np
import pandas as pd

print("Python", sys.version.split()[0], "| NumPy", np.__version__, "| pandas", pd.__version__)


def find_assignment_dir(start):
    """The folder holding data_manifest.json: this one, a parent, or 06/mid-term below one."""
    for folder in (start, *start.parents):
        for candidate in (folder, folder / "06" / "mid-term"):
            if (candidate / "data_manifest.json").is_file():
                return candidate
    raise FileNotFoundError("Open this notebook from the assignment folder that holds data_manifest.json.")


BASE_DIR = find_assignment_dir(Path.cwd().resolve())
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)
PROFILE_PATH = OUTPUT_DIR / "table_profile.csv"
AUDIT_PATH = OUTPUT_DIR / "issue_audit.csv"
PATIENTS_SUMMARY_PATH = OUTPUT_DIR / "patients_summary.csv"
GENDER_CATEGORIES_PATH = OUTPUT_DIR / "gender_categories.csv"
VISITS_SUMMARY_PATH = OUTPUT_DIR / "visits_summary.csv"
DECISION_LOG_PATH = OUTPUT_DIR / "decision_log.csv"
A1C_SUMMARY_PATH = OUTPUT_DIR / "a1c_summary.csv"
MERGE_AUDIT_PATH = OUTPUT_DIR / "merge_audit.csv"
SBP_WIDE_SAMPLE_PATH = OUTPUT_DIR / "sbp_wide_sample.csv"
ANALYSIS_SAMPLE_PATH = OUTPUT_DIR / "analysis_sample.csv"

# The patients whose rows the two extracts hold (README Tasks 4.3 and 5.2).
SAMPLE_IDS = ["P00001", "P00004", "P00088", "P00260", "P00446", "P00487", "P00631", "P00851",
              "P02354", "P03836", "P05763", "P07510", "P14287", "P16486", "P17390", "P30001"]

# The data is generated, not committed: generate_data.py writes the same tables on every computer.
manifest = json.loads((BASE_DIR / "data_manifest.json").read_text(encoding="utf-8"))
if not all((DATA_DIR / name).is_file() for name in manifest["files"]):
    print(subprocess.run([sys.executable, "generate_data.py"], cwd=BASE_DIR, check=True,
                         capture_output=True, text=True).stdout.strip())


def read_raw(path):
    """A Parquet file with the dtypes it stores; a CSV file with every column as text, blanks kept as written."""
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    return pd.read_csv(path, dtype="string", keep_default_na=False)


raw = {Path(name).stem: read_raw(DATA_DIR / name) for name in manifest["files"]}
# The checksum covers each table's contents, written out as CSV text.
for name, expected in manifest["files"].items():
    text = raw[Path(name).stem].to_csv(index=False, lineterminator="\n")
    assert hashlib.sha256(text.encode("utf-8")).hexdigest() == expected["sha256"], (
        f"data/{name} differs from the generated file; delete the data folder and run this cell again.")
print(f"Verified {len(manifest['files'])} data files")
raw_snapshot = {name: table.copy(deep=True) for name, table in raw.items()}
list(raw)

Python 3.13.15 | NumPy 2.3.3 | pandas 3.0.5
Verified 5 data files


['patients', 'sites', 'visits_north', 'visits_south', 'a1c_wide']

## Cumulative midterm checkpoint (human review)

**TODO:** Replace this text with an evidence map: one bullet for each of Lectures 01 to 06. For 01, 02, 04, 05, and 06, name one concrete technique or file and say in one sentence where this notebook or its outputs use it. For 03, complete the NumPy calculation after Task 4.3 and explain your axis choice in one sentence here. The lectures cover the command line and Python basics (01); Git, functions, and files (02); environments, shell pipelines, and NumPy (03); Jupyter and labeled pandas data (04); data contracts, cleaning, and validation (05); and joins, reshaping, and concatenation (06). Do not paste earlier solutions.

Lecture 01:
In task 1.2, I created a function called "profile" that stores a list of dictionaries storing the name, number of rows, number of columns, and number of duplicates for each file stored in "data_manifest.json".

Lecture 02:
In task 1.2, I created appended each file in "data_manifest.json" to my "results" list using "results.append()". 

Lecture 03:

Lecture 04: 
Using ipynb Pandas notebook, I converted a list called "results" into a DataFrame using pd.DataFrame() in task 1.2.

Lecture 05:
Task 2.3 checks to make sure that the visit date is formatted correctly as 'YYYY-MM-DD'. It counts the number of rows without the correct format and saves it in "issue_audit" with "date parse failures." 

Lecture 06:


## Task 1: Inventory the files

### 1.1 Count lines from the terminal

Make `output/file_inventory.txt` in the terminal (README Task 1.1), then paste the command you ran into the `%%bash` cell below.

In [128]:
%%bash
# paste your command here
wc -l data/*.csv > output/file_inventory.txt

### 1.2 Profile each table with a function

In [129]:
# TODO: write a function, such as profile(name, table), that returns the four values in README
# Task 1.2, call it once per name in manifest["files"] with raw[Path(name).stem], and save the results to PROFILE_PATH.
def profile(name, table):
    dict_1 = {"table": name, "rows": table.shape[0], "columns": table.shape[1], "exact_duplicate_rows": table.duplicated().sum()}
    return dict_1

results = list()
for name in manifest["files"]:
    results.append(profile(name, raw[Path(name).stem]))

results = pd.DataFrame(results)
results.to_csv(PROFILE_PATH, index=False)


## Task 2: Contract, stack, and audit

### 2.1 Write the data contract (human review)

**TODO:** Replace this text with the contract: the intended grain and key (single or composite) of each cleaned input table, and the cardinality you expect for each Task 4.4 join. Describe visits after stacking and keeping the later recheck, sites after keeping current records, and the long-form A1c join keys. Note raw exact repeats, rechecks, and site versions that violate the intended cleaned keys (README Task 2.1).

### 2.2 Stack the two visit files

In [130]:
# TODO: rename wt_kg, add source_file to each visit table, and stack them, north first (README Task 2.2).
raw_north = raw["visits_north"].copy(deep = True)
raw_north["source_file"] = "visits_north.csv"

raw_south = raw["visits_south"].copy(deep = True)
raw_south = raw_south.rename(columns = {"wt_kg": "weight_kg"})
raw_south["source_file"] = "visits_south.csv"

visits_raw = pd.concat([raw_north, raw_south], ignore_index = True)

display(visits_raw)

,patient_id,visit,visit_date,visit_time,sbp_mmhg,weight_kg,source_file
0,P00002,baseline,2026-02-20,09:50,132,85.2,visits_north.csv
1,P00002,week12,2026-05-08,15:20,129,82.0,visits_north.csv
2,P00003,baseline,2026-01-16,11:45,148,67.5,visits_north.csv
3,P00003,wk12,2026-04-07,10:00,137,64.0,visits_north.csv
4,P00004,baseline,2026-02-25,10:55,145,106.2,visits_north.csv
...,...,...,...,...,...,...,...
57366,P24976,week12,2026-05-14,10:50,153,78.3,visits_south.csv
57367,P19276,baseline,2026-01-22,15:35,160,122.2,visits_south.csv
57368,P21334,week12,2026-05-05,09:40,129,98.2,visits_south.csv
57369,P03524,baseline,2026-02-06,10:05,160,117.6,visits_south.csv


### 2.3 Audit the raw tables

In [131]:
# TODO: count the 10 issues in README Task 2.3, in order, as a DataFrame with the columns issue and
# count, and save it to AUDIT_PATH.
patients = Path("data") / "patients.parquet"
patients = pd.read_parquet(patients)
sites = Path("data") / "sites.csv"
sites = pd.read_csv(sites)
issue_audit = list()
issue_audit.append(["patients: gender values that are not a code", sum(~(patients["gender"].str.strip().isin(["F", "M", "X"])))])
issue_audit.append(["patients: empty gender", (patients["gender"].str.strip()=="").sum()])
issue_audit.append(["patients: birth_year sentinel 9999", (patients["birth_year"].str.strip()=='9999').sum()])
issue_audit.append(["patients: site codes needing normalization", sum(~(patients["site_code"].str.strip().isin(sites["site_code"])))])
issue_audit.append(["sites: rows with a repeated site_code", sum(sites.duplicated(subset=["site_code"], keep=False))])
issue_audit.append(["visits: visit labels that are not baseline or week12", sum(~((visits_raw["visit"] == "baseline") | (visits_raw["visit"] == "week12")))])

sbps = pd.to_numeric(visits_raw["sbp_mmhg"])
issue_audit.append(["visits: sbp sentinel 999", sum(sbps == 999)])
issue_audit.append(["visits: sbp outside 70 and through 250", sum((sbps < 70)| ((sbps > 250) & ~(sbps == 999)))])
issue_audit.append(["visits: date parse failures", sum(~(visits_raw["visit_date"].str.fullmatch(r'[0-9]{4}-[0-9]{2}-[0-9]{2}')))])
issue_audit.append(["sites: rows in repeated patient_id and visit pairs", sum(visits_raw.duplicated(subset=["patient_id", "visit"], keep=False))])
issue_audit = pd.DataFrame(issue_audit, columns = ["issue", "count"])

display(issue_audit)

assert all(raw[name].equals(raw_snapshot[name]) for name in raw), "Auditing must not change raw."
issue_audit.to_csv(AUDIT_PATH, index = False)

,issue,count
0,patients: gender values that are not a code,7026
1,patients: empty gender,486
2,patients: birth_year sentinel 9999,371
3,patients: site codes needing normalization,1918
4,sites: rows with a repeated site_code,4
5,visits: visit labels that are not baseline or ...,2309
6,visits: sbp sentinel 999,341
7,visits: sbp outside 70 and through 250,116
8,visits: date parse failures,77
9,sites: rows in repeated patient_id and visit p...,288


## Task 3: Clean

### 3.1 Clean the patients

In [132]:
# TODO: list the gender spellings with value_counts(), then build patients_clean from
# raw["patients"].copy(deep=True) with steps 1 to 5 in README Task 3.1.
patients_task3 = raw["patients"].copy(deep=True)
patients_task3 = patients_task3.drop_duplicates(subset = None, keep='first')
gender_counts = patients_task3["gender"].value_counts()
patients_task3["gender"] = patients_task3["gender"].str.strip().str.lower().map({"f": "F", "female": "F", "woman": "F", "m": "M", "male": "M", "man": "M",
                                                                       "x": "X", "nb": "X", "non-binary": "X"})
gender_counts = patients_task3["gender"].value_counts()
patients_task3["gender"] = patients_task3["gender"].astype("category")
patients_task3["birth_year"] = patients_task3["birth_year"].map(lambda x: np.nan if x == "9999" else x)
patients_task3["site_code"] = patients_task3["site_code"].str.strip().str.upper().str.replace("-", "").str.replace(" ", "")

patients_clean = patients_task3.reset_index(drop=True)
display(patients_clean)
display(patients_clean["gender"].value_counts())

,patient_id,gender,birth_year,site_code
0,P00001,F,1966,S02
1,P00002,F,NaN,N01
2,P00003,F,1970,N01
3,P00004,X,NaN,N01
4,P00005,M,1947,N01
...,...,...,...,...
29995,P29996,F,1967,S02
29996,P29997,F,1970,N01
29997,P29998,NaN,1967,N01
29998,P29999,M,1956,S02


gender
F    14413
M    13955
X      651
Name: count, dtype: int64

In [133]:
# TODO: show patients_clean["gender"].cat.categories, save the patients summary to
# PATIENTS_SUMMARY_PATH, and save the gender value_counts(dropna=False) with a code column from
# .cat.codes to GENDER_CATEGORIES_PATH, as in README Task 3.1.

display(patients_clean["gender"].cat.categories)

summary = pd.DataFrame({
    "metric": ["patients", "birth_year missing", "birth_year min", "birth_year max", "site_code values", "patients at N01", "patients at S02", "patients at E04"],
    "value": [len(patients_clean), patients_clean["birth_year"].isna().sum(), patients_clean["birth_year"].min(), patients_clean["birth_year"].max(), patients_clean["site_code"].nunique(), (patients_clean["site_code"] == "N01").sum(),
              (patients_clean["site_code"] == "S02").sum(), (patients_clean["site_code"] == "E04").sum()],
})

summary.to_csv(PATIENTS_SUMMARY_PATH, index = False)

gender_counts = patients_clean["gender"].value_counts(dropna=False).reset_index()
gender_counts["code"] = gender_counts["gender"].cat.codes
gender_counts.to_csv(GENDER_CATEGORIES_PATH, index=False)

Index(['F', 'M', 'X'], dtype='str')

### 3.2 Clean the visits

In [134]:
# TODO: build visits_clean from visits_raw.copy(deep=True) with steps 1 to 5 in README Task 3.2.
visits_task3 = visits_raw.copy(deep=True)
visits_task3 = visits_task3.drop_duplicates(subset = None, keep = "first")
visit_counts = visits_task3["visit"].value_counts()
visits_task3["visit"] = visits_task3["visit"].str.strip().str.lower().map({"baseline": "baseline", "bl": "baseline", "week12": "week12", "wk12": "week12", "week 12": "week12"})
visit_counts = visits_task3["visit"].value_counts()
visits_task3["sbp_mmhg"] = pd.to_numeric(visits_task3["sbp_mmhg"], errors = "coerce")
visits_task3["sbp_mmhg"] = visits_task3["sbp_mmhg"].map(lambda x: np.nan if (x<70)|(x>250) else x)
visits_task3["weight_kg"] = pd.to_numeric(visits_task3["weight_kg"], errors = "coerce")

visit_date_match = visits_task3["visit_date"].str.fullmatch(r'[0-9]{4}-[0-9]{2}-[0-9]{2}')
visit_date_all = visits_task3["visit_date"]
visits_task3["visit_date"] = visit_date_all[visit_date_match]
visits_clean = visits_task3
display(visits_clean)

#patients_task3["birth_year"] = patients_task3["birth_year"].map(lambda x: np.nan if x == "9999" else x)
#patients_task3["site_code"] = patients_task3["site_code"].str.strip().str.upper().str.replace("-", "").str.replace(" ", "")

#patients_clean = patients_task3.reset_index(drop=True)
#display(visits_task3)
#display(patients_clean["gender"].value_counts())

,patient_id,visit,visit_date,visit_time,sbp_mmhg,weight_kg,source_file
0,P00002,baseline,2026-02-20,09:50,132.0,85.2,visits_north.csv
1,P00002,week12,2026-05-08,15:20,129.0,82.0,visits_north.csv
2,P00003,baseline,2026-01-16,11:45,148.0,67.5,visits_north.csv
3,P00003,week12,2026-04-07,10:00,137.0,64.0,visits_north.csv
4,P00004,baseline,2026-02-25,10:55,145.0,106.2,visits_north.csv
...,...,...,...,...,...,...,...
57331,P30038,week12,2026-04-20,09:05,135.0,109.0,visits_south.csv
57332,P30039,baseline,2026-02-24,10:10,142.0,81.8,visits_south.csv
57333,P30039,week12,2026-05-15,10:05,138.0,80.3,visits_south.csv
57334,P30040,baseline,2026-02-04,14:40,144.0,121.0,visits_south.csv


In [135]:
# TODO: step 6, show the repeated patient_id and visit pairs, then keep the later reading.
display(visits_task3[visits_task3.duplicated(subset = ["patient_id", "visit"], keep = False)])
display(visits_task3.sort_values("visit_time").drop_duplicates(subset = ["patient_id", "visit"], keep = "last"))

,patient_id,visit,visit_date,visit_time,sbp_mmhg,weight_kg,source_file
4,P00004,baseline,2026-02-25,10:55,145.0,106.2,visits_north.csv
5,P00004,baseline,2026-02-25,11:15,142.0,106.2,visits_north.csv
84,P00088,baseline,2026-03-10,09:30,153.0,68.2,visits_north.csv
85,P00088,baseline,2026-03-10,09:50,152.0,68.2,visits_north.csv
1208,P01341,week12,2026-05-26,15:10,121.0,72.3,visits_north.csv
...,...,...,...,...,...,...,...
55282,P28051,week12,2026-04-16,09:35,126.0,92.7,visits_south.csv
55472,P28232,baseline,2026-01-12,10:00,153.0,61.0,visits_south.csv
55473,P28232,baseline,2026-01-12,10:10,159.0,61.0,visits_south.csv
56248,P28953,baseline,2026-01-21,10:45,151.0,74.8,visits_south.csv


,patient_id,visit,visit_date,visit_time,sbp_mmhg,weight_kg,source_file
122,P00137,week12,2026-06-15,08:00,128.0,66.9,visits_north.csv
1190,P01321,week12,2026-06-03,08:00,127.0,96.3,visits_north.csv
1357,P01507,baseline,2026-01-28,08:00,140.0,102.1,visits_north.csv
3726,P04163,baseline,2026-01-16,08:00,127.0,110.8,visits_north.csv
4648,P05164,week12,2026-05-18,08:00,134.0,69.1,visits_north.csv
...,...,...,...,...,...,...,...
54017,P26802,baseline,2026-01-06,16:50,159.0,103.1,visits_south.csv
54337,P27087,week12,2026-05-20,16:50,130.0,77.0,visits_south.csv
54490,P27247,baseline,2026-03-20,16:50,136.0,91.2,visits_south.csv
54562,P27316,week12,2026-04-13,16:50,125.0,<NA>,visits_south.csv


In [136]:
# TODO: step 7, sort, renumber, keep the six columns, then save the visits summary to
# VISITS_SUMMARY_PATH (README Task 3.2).
visits_clean = visits_clean.sort_values(["patient_id", "visit"]).reset_index(drop=True)
visits_clean = visits_clean.drop(columns = "visit_time")

visits_summary = pd.DataFrame({
    "metric": ["visits", "baseline visits", "week12 visits", "visit_date missing", "visit_date min", "visit_date max", "sbp_mmhg missing", "sbp_mmhg min", "sbp_mmhg max", "weight_kg missing", "weight_kg mean", "visits from visits_north.csv", "visits from visits_south.csv"],
    "value": [len(visits_clean), sum(visits_clean["visit"]=="baseline"), sum(visits_clean["visit"]=="week12"), visits_clean["visit_date"].isna().sum(), visits_clean["visit_date"].min(), visits_clean["visit_date"].max(),
              visits_clean["sbp_mmhg"].isna().sum(), visits_clean["sbp_mmhg"].min(), visits_clean["sbp_mmhg"].max(), visits_clean["weight_kg"].isna().sum(), visits_clean["weight_kg"].mean(), sum(visits_clean["source_file"]=="visits_north.csv"), sum(visits_clean["source_file"]=="visits_south.csv")],
})
visits_summary.to_csv(VISITS_SUMMARY_PATH, index = False)

### 3.3 Record the decisions

In [ ]:
# TODO: one row per issue in README Task 3.3, with table and issue copied, and your own
# decision and one-sentence reason, then save it to DECISION_LOG_PATH.
decision_table = list()
decision_table.append(["patients", "exact duplicate rows", "dropped exact duplicates, keeping only the first one", "duplicates were dropped so that only one visit is recorded per row and avoiding confusion with writing the same visit multiple times"])
decision_table.append(["patients", "gender spellings that are not a code", "removed excess whitespace from gender data entries using str.strip(), lowercased my entries, and mapped my entries to display only M, F, or X", "stripping and lower-casing my entries simplifies the mapping process and standardizes my patient dataset so that there is only one letter for each gender"])
decision_table.append(["patients", "site codes needing normalization", "hyphens and dashes were removed from the site codes and letters were upper-cased", "these letter/symbol adjustments were made to ensure all site codes follow the same format and to ensure that each code is not written in multiple different formats, standardizing my patient dataset and making it easier to evaluate"])
decision_table = pd.DataFrame(decision_table, columns=["table", "issue", "decision", "reason"])
display(decision_table)

,table,issue,decision,reason
0,patients,exact duplicate rows,"dropped exact duplicates, keeping only the fir...",duplicates were dropped so that only one visit...
1,patients,gender spellings that are not a code,removed excess whitespace from gender data ent...,stripping and lower-casing my entries simplifi...


## Task 4: Reshape and join

### 4.1 Melt the A1c results to long form

In [138]:
# TODO: melt raw["a1c_wide"], map the visit labels, drop the rows with no result,
# then save the per-visit summary to A1C_SUMMARY_PATH (README Task 4.1).
a1c_long = None

### 4.2 Check the keys before joining

In [139]:
# TODO: steps 1 and 2, catch the MergeError, show the repeated site rows, and build sites_current.
sites = raw["sites"]
sites_current = None

In [140]:
# TODO: step 3, outer merge visits_clean and a1c_long with indicator=True and show the _merge counts,
# then save the seven counts in README Task 4.2 to MERGE_AUDIT_PATH.

### 4.3 Pivot blood pressure to one row per patient

In [141]:
# TODO: pivot visits_clean, reset the index, clear columns.name, add sbp_change, then save the
# SAMPLE_IDS rows to SBP_WIDE_SAMPLE_PATH (README Task 4.3).
sbp_wide = None

#### NumPy checkpoint (Lecture 03, human review)

From the full `sbp_wide`, select `baseline` and `week12` in that order, keeping only patients with both readings. Name this two-column table `complete_pairs`; `np.array(complete_pairs, dtype=float)` converts it to a NumPy array. Display its shape and calculate one NumPy mean per visit across patients. Save the outputs and explain your axis choice in the cumulative checkpoint's Lecture 03 bullet. No extra output file.

In [142]:
# TODO: select complete pairs, convert to a NumPy array, and display its shape and two per-visit means.

### 4.4 Join the analysis table

In [143]:
# TODO: three left merges onto visits_clean, each with validate=, then the columns and dtypes
# in README Task 4.4. A MergeError means an earlier step left a problem: see HINTS.md.
analysis = None

## Task 5: Validate, save, and read back

### 5.1 Validate before saving

In [144]:
# TODO: collect named True/False checks in a Series (README Task 5.1), display it, and stop with
# assert checks.all(), checks[~checks] before the analysis extract is saved.
# A failed check means an earlier step left a problem: print checks[~checks] and see HINTS.md.

### 5.2 Save the analysis extract

In [145]:
# TODO: keep the analysis rows whose patient_id is in SAMPLE_IDS, renumber them, and save them to
# ANALYSIS_SAMPLE_PATH with index=False (README Task 5.2).

### 5.3 Read the file back

In [146]:
# TODO: read ANALYSIS_SAMPLE_PATH back with its dtypes and parse_dates=["visit_date"], and print
# whether it equals analysis_sample.